In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt

print("Libraries loaded")

Libraries loaded


In [2]:
label_map = {0: 'Other', 1: 'Ruminating', 2: 'Eating'}

def load_and_merge(accel_path, halter_path):
    accel  = pd.read_csv(accel_path, 
                         skiprows=lambda x: x % 10 != 0 and x != 0)
    halter = pd.read_csv(halter_path)
    
    accel['timestamp']  = pd.to_datetime(accel['timestamp']).dt.round('1s')
    halter['timestamp'] = pd.to_datetime(halter['timestamp']).dt.round('1s')
    
    merged = pd.merge(accel, halter, on='timestamp', how='inner')
    merged['behavior'] = merged['classification'].map(label_map)
    merged = merged.drop(columns=['classification'])
    return merged

print("Loading cow_01...")
cow01 = load_and_merge('../data/accel-01.csv', '../data/halter-01.csv')
print(f"  cow_01: {len(cow01):,} rows")

print("Loading cow_03...")
cow03 = load_and_merge('../data/accel-03.csv', '../data/halter-03.csv')
print(f"  cow_03: {len(cow03):,} rows")

print("Loading cow_04...")
cow04 = load_and_merge('../data/accel-04.csv', '../data/halter-04.csv')
print(f"  cow_04: {len(cow04):,} rows")

print("\nAll animals loaded")

Loading cow_01...
  cow_01: 14,595,882 rows
Loading cow_03...
  cow_03: 11,483,816 rows
Loading cow_04...
  cow_04: 5,060,284 rows

All animals loaded


In [3]:
def extract_windows(df, animal_id, window_size=30):
    """
    Splits time-series data into non-overlapping windows.
    Each window = 30 seconds of data = 30 rows (at 1Hz after sampling).
    Extracts statistical features from x, y, z per window.
    Assigns majority behavior label to each window.
    """
    windows = []
    n = len(df)
    
    for start in range(0, n - window_size, window_size):
        window = df.iloc[start:start + window_size]
        
        x = window['x'].values
        y = window['y'].values
        z = window['z'].values
        
        # Signal magnitude (overall movement intensity)
        magnitude = np.sqrt(x**2 + y**2 + z**2)
        
        # ODBA - Overall Dynamic Body Acceleration
        # Standard metric in animal movement ecology
        odba = (np.abs(x - x.mean()) + 
                np.abs(y - y.mean()) + 
                np.abs(z - z.mean()))
        
        # Majority label for this window
        majority_label = window['behavior'].mode()[0]
        
        features = {
            'animal_id':    animal_id,
            'window_start': window['timestamp'].iloc[0],
            
            # X features
            'x_mean':  x.mean(),
            'x_std':   x.std(),
            'x_min':   x.min(),
            'x_max':   x.max(),
            'x_range': x.max() - x.min(),
            
            # Y features
            'y_mean':  y.mean(),
            'y_std':   y.std(),
            'y_min':   y.min(),
            'y_max':   y.max(),
            'y_range': y.max() - y.min(),
            
            # Z features
            'z_mean':  z.mean(),
            'z_std':   z.std(),
            'z_min':   z.min(),
            'z_max':   z.max(),
            'z_range': z.max() - z.min(),
            
            # Combined features
            'magnitude_mean': magnitude.mean(),
            'magnitude_std':  magnitude.std(),
            'odba_mean':      odba.mean(),
            'odba_std':       odba.std(),
            
            # Label
            'behavior': majority_label,
        }
        
        windows.append(features)
    
    return pd.DataFrame(windows)

print("Windowing function defined")

Windowing function defined


In [4]:
print("Extracting windows...")

windows_01 = extract_windows(cow01, 'cow_01')
print(f"  cow_01: {len(windows_01):,} windows")

windows_03 = extract_windows(cow03, 'cow_03')
print(f"  cow_03: {len(windows_03):,} windows")

windows_04 = extract_windows(cow04, 'cow_04')
print(f"  cow_04: {len(windows_04):,} windows")

print("\nFirst 3 windows of cow_01:")
print(windows_01.head(3))

Extracting windows...
  cow_01: 486,529 windows
  cow_03: 382,793 windows
  cow_04: 168,676 windows

First 3 windows of cow_01:
  animal_id        window_start      x_mean       x_std  x_min  x_max  \
0    cow_01 2015-06-12 13:30:01  134.800000   31.549326     20    168   
1    cow_01 2015-06-12 13:30:04 -243.066667  212.320973   -488     20   
2    cow_01 2015-06-12 13:30:07 -752.533333   90.011752   -880   -672   

   x_range      y_mean      y_std  y_min  ...      z_mean      z_std  z_min  \
0      148  612.933333  17.879100    592  ...  847.866667  11.146699    840   
1      508  562.666667  68.221860    476  ...  859.866667  82.865245    768   
2      208  416.666667  83.151802    304  ...  624.133333  84.490525    504   

   z_max  z_range  magnitude_mean  magnitude_std   odba_mean   odba_std  \
0    864       24     1055.510932       8.493481   51.137778  20.216773   
1    972      204     1080.212972      69.187202  310.542222  94.665959   
2    712      208     1073.023559    

In [5]:
print("=== Label distribution in windowed data ===\n")

for name, df in [('cow_01', windows_01), 
                 ('cow_03', windows_03), 
                 ('cow_04', windows_04)]:
    total = len(df)
    print(f"{name}:")
    for b in ['Eating', 'Ruminating', 'Other']:
        count = len(df[df['behavior'] == b])
        print(f"  {b:12s}: {count:6,} windows ({count/total*100:.1f}%)")
    print()

=== Label distribution in windowed data ===

cow_01:
  Eating      : 94,175 windows (19.4%)
  Ruminating  : 195,232 windows (40.1%)
  Other       : 197,122 windows (40.5%)

cow_03:
  Eating      : 81,118 windows (21.2%)
  Ruminating  : 142,223 windows (37.2%)
  Other       : 159,452 windows (41.7%)

cow_04:
  Eating      : 28,887 windows (17.1%)
  Ruminating  : 61,149 windows (36.3%)
  Other       : 78,640 windows (46.6%)



In [6]:
# Save individual animal windows
windows_01.to_csv('../data/windows_cow01.csv', index=False)
windows_03.to_csv('../data/windows_cow03.csv', index=False)
windows_04.to_csv('../data/windows_cow04.csv', index=False)

# Save training set (cow_01 + cow_03 combined)
# cow_04 is held out for anomaly detection testing
train_windows = pd.concat([windows_01, windows_03], ignore_index=True)
train_windows.to_csv('../data/windows_train.csv', index=False)

print(f"Training windows (cow_01 + cow_03): {len(train_windows):,}")
print(f"Test windows    (cow_04):           {len(windows_04):,}")
print("\nAll window files saved to ../data/")
print("\nFeature columns:")
feature_cols = [c for c in windows_01.columns 
                if c not in ['animal_id', 'window_start', 'behavior']]
print(feature_cols)
print(f"\nTotal features: {len(feature_cols)}")

Training windows (cow_01 + cow_03): 869,322
Test windows    (cow_04):           168,676

All window files saved to ../data/

Feature columns:
['x_mean', 'x_std', 'x_min', 'x_max', 'x_range', 'y_mean', 'y_std', 'y_min', 'y_max', 'y_range', 'z_mean', 'z_std', 'z_min', 'z_max', 'z_range', 'magnitude_mean', 'magnitude_std', 'odba_mean', 'odba_std']

Total features: 19


In [7]:
sample = cow01.head(100)
print("First 10 timestamps in raw merged data:")
print(sample['timestamp'].head(10).tolist())
print(f"\nTime between rows: {sample['timestamp'].diff().dropna().mode()[0]}")
print(f"Window size = 30 rows")
print(f"Window duration = 30 × time_between_rows")

First 10 timestamps in raw merged data:
[Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:01'), Timestamp('2015-06-12 13:30:02')]

Time between rows: 0 days 00:00:00
Window size = 30 rows
Window duration = 30 × time_between_rows


In [8]:
# Check how many duplicates exist
print("Duplicate timestamps in cow01:")
print(f"  Total rows:      {len(cow01):,}")
print(f"  Unique timestamps: {cow01['timestamp'].nunique():,}")
print(f"  Duplicates:      {len(cow01) - cow01['timestamp'].nunique():,}")

# Keep first occurrence of each timestamp
cow01_dedup = cow01.drop_duplicates(subset='timestamp', keep='first').reset_index(drop=True)
cow03_dedup = cow03.drop_duplicates(subset='timestamp', keep='first').reset_index(drop=True)
cow04_dedup = cow04.drop_duplicates(subset='timestamp', keep='first').reset_index(drop=True)

print(f"\nAfter deduplication:")
print(f"  cow01: {len(cow01_dedup):,} rows")
print(f"  cow03: {len(cow03_dedup):,} rows")
print(f"  cow04: {len(cow04_dedup):,} rows")

# Verify time between rows now
print(f"\nTime between rows after dedup:")
print(cow01_dedup['timestamp'].diff().dropna().value_counts().head(5))

Duplicate timestamps in cow01:
  Total rows:      14,595,882
  Unique timestamps: 1,450,800
  Duplicates:      13,145,082

After deduplication:
  cow01: 1,450,800 rows
  cow03: 1,128,599 rows
  cow04: 506,028 rows

Time between rows after dedup:
timestamp
0 days 00:00:01    1450799
Name: count, dtype: int64


In [9]:
print("Re-extracting windows on deduplicated data...")
print("Window size = 30 rows = ~30 seconds (1 row per second)")

windows_01 = extract_windows(cow01_dedup, 'cow_01', window_size=30)
windows_03 = extract_windows(cow03_dedup, 'cow_03', window_size=30)
windows_04 = extract_windows(cow04_dedup, 'cow_04', window_size=30)

print(f"\n  cow_01: {len(windows_01):,} windows")
print(f"  cow_03: {len(windows_03):,} windows")
print(f"  cow_04: {len(windows_04):,} windows")

# Verify window duration
w0_start = windows_01['window_start'].iloc[0]
w1_start = windows_01['window_start'].iloc[1]
print(f"\nTime between window starts: {w1_start - w0_start}")
print("(should be ~30 seconds)")

Re-extracting windows on deduplicated data...
Window size = 30 rows = ~30 seconds (1 row per second)

  cow_01: 48,359 windows
  cow_03: 37,619 windows
  cow_04: 16,867 windows

Time between window starts: 0 days 00:00:30
(should be ~30 seconds)


In [10]:
print("=== Label distribution after fix ===\n")

for name, df in [('cow_01', windows_01),
                 ('cow_03', windows_03),
                 ('cow_04', windows_04)]:
    total = len(df)
    print(f"{name}:")
    for b in ['Eating', 'Ruminating', 'Other']:
        count = len(df[df['behavior'] == b])
        print(f"  {b:12s}: {count:6,} windows ({count/total*100:.1f}%)")
    print()

# Resave everything
windows_01.to_csv('../data/windows_cow01.csv', index=False)
windows_03.to_csv('../data/windows_cow03.csv', index=False)
windows_04.to_csv('../data/windows_cow04.csv', index=False)

train_windows = pd.concat([windows_01, windows_03], ignore_index=True)
train_windows.to_csv('../data/windows_train.csv', index=False)

print(f"Training windows (cow_01 + cow_03): {len(train_windows):,}")
print(f"Test windows     (cow_04):          {len(windows_04):,}")
print("\nFiles resaved successfully")

=== Label distribution after fix ===

cow_01:
  Eating      :  9,460 windows (19.6%)
  Ruminating  : 19,277 windows (39.9%)
  Other       : 19,622 windows (40.6%)

cow_03:
  Eating      :  7,871 windows (20.9%)
  Ruminating  : 14,053 windows (37.4%)
  Other       : 15,695 windows (41.7%)

cow_04:
  Eating      :  2,859 windows (17.0%)
  Ruminating  :  6,136 windows (36.4%)
  Other       :  7,872 windows (46.7%)

Training windows (cow_01 + cow_03): 85,978
Test windows     (cow_04):          16,867

Files resaved successfully
